In [1]:
!pip install -q nltk

import os, re, zipfile, collections, pickle
from collections import defaultdict
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import matplotlib.pyplot as plt

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
BASE = "/content"                                  # keep this as /content in Colab
ZIP_PATH = f"{BASE}/archive (6) (1).zip"           # the name Colab saved your upload as
if not os.path.exists(ZIP_PATH):
    from google.colab import files
    up = files.upload()
    ZIP_PATH = f"{BASE}/" + list(up)[0]

DATA_DIR = f"{BASE}/cranfield_data"
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATA_DIR)

# documents: doc_id (str "1".."1400") -> text (title + abstract in one file)
docs = {}
for i in range(1, 1401):
    with open(f"{DATA_DIR}/Cranfield/{i}.txt", encoding="utf-8", errors="replace") as f:
        docs[str(i)] = f.read().strip()

# queries: query_id (str) -> text
queries = {}
with open(f"{DATA_DIR}/TEST/query.txt", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.strip()
        if line:
            qid, text = line.split("\t", 1)
            queries[qid] = text.strip()

# answer key: qrels[query_id][doc_id] = grade  (lines look like "1 184<TAB>2")
qrels = defaultdict(dict)
skipped = []
for qid in queries:
    with open(f"{DATA_DIR}/TEST/RES/{qid}.txt", encoding="utf-8", errors="replace") as f:
        for line in f:
            parts = line.split()
            if not line.strip():
                continue
            if len(parts) != 3:                      # malformed line (e.g. query 40: "40 85" has no grade)
                skipped.append((qid, line.strip()))
                continue
            q, d, g = parts
            qrels[q][d] = int(g)

print("docs:", len(docs), "| queries:", len(queries), "| queries with qrels:", len(qrels))
print("skipped malformed answer-key lines:", skipped)
print("grades present:", sorted(collections.Counter(g for q in qrels.values() for g in q.values()).items()))
print("\nSample doc:", docs["1"][:200], "...")
print("Sample query:", queries["1"])

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
qrels_gain = {q: {d: 5 - g for d, g in dd.items() if g > 0} for q, dd in qrels.items()}

n_rel = sum(len(dd) for dd in qrels_gain.values())
print("Relevant judgments (grades 1-4):", n_rel)
print("Queries with at least one relevant doc:", sum(1 for dd in qrels_gain.values() if dd), "of", len(queries))
print("Avg relevant docs per query:", round(n_rel / len(queries), 1))

In [ ]:
STOP = set(stopwords.words("english"))
stemmer = PorterStemmer()

def clean_raw(text):
    """Light cleaning for the Sentence Transformer."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def preprocess(text):
    """Heavy cleaning for TF-IDF/BM25: lowercase, clean, tokenize, stop-words, stem."""
    text = re.sub(r"[^a-z\s]", " ", text.lower())                    # lowercase + drop punctuation/numbers
    tokens = text.split()                                            # tokenize
    tokens = [t for t in tokens if t not in STOP and len(t) > 1]     # remove stop-words
    return [stemmer.stem(t) for t in tokens]                         # stem

example = "What are the effects of heat on supersonic flow?"
print("Before:", example)
print("After :", preprocess(example))

In [ ]:
doc_tokens    = {d: preprocess(t) for d, t in docs.items()}
query_tokens  = {q: preprocess(t) for q, t in queries.items()}
docs_clean    = {d: clean_raw(t) for d, t in docs.items()}      # for the dense model
queries_clean = {q: clean_raw(t) for q, t in queries.items()}

print("Docs with empty token list:", sum(len(t) == 0 for t in doc_tokens.values()))
print("Queries with empty token list:", sum(len(t) == 0 for t in query_tokens.values()))
print("\nDoc 1 tokens:", doc_tokens["1"][:15])

In [ ]:
def build_index(doc_tokens):
    inv = defaultdict(dict)                      # term -> {doc_id: term_frequency}
    for d, toks in doc_tokens.items():
        for t in toks:
            inv[t][d] = inv[t].get(d, 0) + 1
    return inv

inv_index = build_index(doc_tokens)
doc_len   = {d: len(t) for d, t in doc_tokens.items()}
avgdl     = sum(doc_len.values()) / len(doc_len)
df        = {t: len(p) for t, p in inv_index.items()}    # document frequency

print(len(inv_index), "unique terms | average doc length:", round(avgdl, 1))
term = "flow"
print(f"Posting list for '{term}': appears in {df.get(term, 0)} docs; first 5:", list(inv_index.get(term, {}).items())[:5])

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist([len(t.split()) for t in docs.values()], bins=40, color="#3730a3")
plt.xlabel("Words per document"); plt.ylabel("Number of documents")
plt.title("Cranfield document length distribution")
plt.tight_layout(); plt.savefig("doc_lengths.png", dpi=200); plt.show()

qid = "1"
best = sorted(qrels_gain[qid].items(), key=lambda x: -x[1])
print("Example query", qid, ":", queries[qid])
print("Relevant docs (doc_id, gain; 4 = best):", best)
print("\nMost relevant document:\n", docs[best[0][0]][:500])

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

SAVE = "/content/drive/MyDrive/IR_project/data.pkl"      # CHANGE: folder shared with the group
os.makedirs(os.path.dirname(SAVE), exist_ok=True)

pickle.dump(dict(docs=docs, queries=queries, qrels=dict(qrels), qrels_gain=qrels_gain,
                 doc_tokens=doc_tokens, query_tokens=query_tokens,
                 docs_clean=docs_clean, queries_clean=queries_clean,
                 inv_index=dict(inv_index), doc_len=doc_len, avgdl=avgdl, df=df),
            open(SAVE, "wb"))
print("Saved to", SAVE)

Person 2: TF-IDF and BM25


In [ ]:
!pip install -q rank_bm25
import numpy as np, pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

K1, B, TOP_K = 1.5, 0.75, 100
doc_ids = list(doc_tokens.keys())
print(len(doc_ids), "documents ready")      # expect 1400

In [ ]:
class TfidfRanker:
    def __init__(self):
        self.vec = TfidfVectorizer(analyzer=lambda toks: toks)
        self.X = self.vec.fit_transform([doc_tokens[d] for d in doc_ids])
    def search_tokens(self, q_tokens, top_k=TOP_K):
        sims = cosine_similarity(self.vec.transform([q_tokens]), self.X).ravel()
        idx = np.argsort(-sims, kind="stable")[:top_k]
        return [(doc_ids[i], float(sims[i])) for i in idx]
    def search(self, query_text, top_k=TOP_K):
        return self.search_tokens(preprocess(query_text), top_k)

tfidf = TfidfRanker()
print("TF-IDF matrix:", tfidf.X.shape)      # expect (1400, about 4275)

In [ ]:
class BM25Ranker:
    def __init__(self, k1=K1, b=B):
        self.bm25 = BM25Okapi([doc_tokens[d] for d in doc_ids], k1=k1, b=b)
    def search_tokens(self, q_tokens, top_k=TOP_K):
        scores = self.bm25.get_scores(q_tokens)
        idx = np.argsort(-scores, kind="stable")[:top_k]
        return [(doc_ids[i], float(scores[i])) for i in idx]
    def search(self, query_text, top_k=TOP_K):
        return self.search_tokens(preprocess(query_text), top_k)

bm25 = BM25Ranker()
q = "1"
print(queries[q])
print("TF-IDF:", tfidf.search(queries[q], 5))
print("BM25  :", bm25.search(queries[q], 5))

In [ ]:
runs = {
  "tfidf": {q: tfidf.search_tokens(query_tokens[q]) for q in queries},
  "bm25":  {q: bm25.search_tokens(query_tokens[q])  for q in queries},
}

def p10(r):  return np.mean([sum(d in qrels_gain[q] for d,_ in r[q][:10])/10 for q in r])
def r100(r): return np.mean([sum(d in qrels_gain[q] for d,_ in r[q])/len(qrels_gain[q]) for q in r])
def mapk(r):
    out = []
    for q in r:
        hits = s = 0
        for i, (d, _) in enumerate(r[q], 1):
            if d in qrels_gain[q]: hits += 1; s += hits/i
        out.append(s/len(qrels_gain[q]))
    return np.mean(out)
def ndcg10(r):
    out = []
    for q in r:
        g = qrels_gain[q]
        dcg  = sum(g.get(d,0)/np.log2(i+1) for i,(d,_) in enumerate(r[q][:10],1))
        idcg = sum(x/np.log2(i+1) for i,x in enumerate(sorted(g.values(), reverse=True)[:10],1))
        out.append(dcg/idcg if idcg else 0)
    return np.mean(out)

for m, r in runs.items():
    print(f"{m:6} P@10={p10(r):.3f}  R@100={r100(r):.3f}  MAP={mapk(r):.3f}  nDCG@10={ndcg10(r):.3f}  queries={len(r)}")

In [ ]:
OUT = "/content/drive/MyDrive/IR_project/sparse_runs.pkl"
pickle.dump(runs, open(OUT, "wb"))
print("Saved", OUT)

### Person 3: Dense retrieval and Reciprocal Rank Fusion (RRF)

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
print(f"Using device: {device}")

In [ ]:
import pickle
import os

DRIVE_DIR = "/content/drive/MyDrive/IR_project"
os.makedirs(DRIVE_DIR, exist_ok=True)
DATA_PATH = os.path.join(DRIVE_DIR, "data.pkl")

# Safely load preprocess variables from memory or drive fallback
if 'docs_clean' not in globals() or 'queries_clean' not in globals():
    if os.path.exists(DATA_PATH):
        print("Loading dataset from Google Drive...")
        with open(DATA_PATH, "rb") as f:
            saved_data = pickle.load(f)
            docs_clean = saved_data["docs_clean"]
            queries_clean = saved_data["queries_clean"]
    else:
        raise NameError("Clean variables not found. Please run the preprocessing cells starting from cell 03ac417e first.")

doc_ids = list(docs_clean.keys())
doc_texts = [docs_clean[d] for d in doc_ids]

print("Encoding Cranfield corpus documents...")
doc_embeddings = model.encode(doc_texts, show_progress_bar=True, convert_to_numpy=True)

query_ids = list(queries_clean.keys())
query_texts = [queries_clean[q] for q in query_ids]

print("Encoding queries...")
query_embeddings = model.encode(query_texts, show_progress_bar=True, convert_to_numpy=True)

print("Document embeddings shape:", doc_embeddings.shape)
print("Query embeddings shape:", query_embeddings.shape)

In [ ]:
DENSE_EMB_OUT = "/content/drive/MyDrive/IR_project/dense_embeddings.pkl"
os.makedirs(os.path.dirname(DENSE_EMB_OUT), exist_ok=True)
with open(DENSE_EMB_OUT, "wb") as f:
    pickle.dump({
        "doc_ids": doc_ids,
        "doc_embeddings": doc_embeddings,
        "query_ids": query_ids,
        "query_embeddings": query_embeddings
    }, f)
print("Saved dense embeddings to:", DENSE_EMB_OUT)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Ensure TOP_K is defined
if 'TOP_K' not in globals():
    TOP_K = 100

class DenseRanker:
    def __init__(self, doc_ids, doc_embeddings):
        self.doc_ids = doc_ids
        self.doc_embeddings = doc_embeddings

    def search_emb(self, q_emb, top_k=100):
        sims = cosine_similarity(q_emb.reshape(1, -1), self.doc_embeddings).ravel()
        idx = np.argsort(-sims, kind="stable")[:top_k]
        return [(self.doc_ids[i], float(sims[i])) for i in idx]

dense_ranker = DenseRanker(doc_ids, doc_embeddings)

dense_runs = {}
for qid, q_emb in zip(query_ids, query_embeddings):
    dense_runs[qid] = dense_ranker.search_emb(q_emb, top_k=TOP_K)

# Ensure runs dict exists
if 'runs' not in globals():
    runs = {}
runs["dense"] = dense_runs

In [ ]:
def rrf(run_list, k=60, top_k=100):
    fused_run = {}
    all_queries = run_list[0].keys()

    for q in all_queries:
        doc_scores = defaultdict(float)
        for run in run_list:
            for rank, (doc_id, _) in enumerate(run[q], start=1):
                doc_scores[doc_id] += 1.0 / (k + rank)

        sorted_docs = sorted(doc_scores.items(), key=lambda x: -x[1])[:top_k]
        fused_run[q] = sorted_docs
    return fused_run

In [ ]:
!pip install -q rank_bm25

import pickle
import numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

DRIVE_DIR = "/content/drive/MyDrive/IR_project"
os.makedirs(DRIVE_DIR, exist_ok=True)
SPARSE_RUNS_PATH = os.path.join(DRIVE_DIR, "sparse_runs.pkl")

loaded_successfully = False
if 'runs' not in globals() or "bm25" not in runs:
    if os.path.exists(SPARSE_RUNS_PATH):
        print("Loading sparse runs from Drive...")
        try:
            with open(SPARSE_RUNS_PATH, "rb") as f:
                runs = pickle.load(f)
            loaded_successfully = True
        except (EOFError, pickle.UnpicklingError):
            print("sparse_runs.pkl on Drive is empty or corrupted. Recomputing...")

    if not loaded_successfully:
        print("Computing TF-IDF and BM25 runs on-the-fly...")
        if 'doc_tokens' not in globals():
            with open(os.path.join(DRIVE_DIR, "data.pkl"), "rb") as f:
                saved = pickle.load(f)
                doc_tokens = saved["doc_tokens"]
                query_tokens = saved["query_tokens"]
                qrels_gain = saved["qrels_gain"]

        doc_ids = list(doc_tokens.keys())

        vec = TfidfVectorizer(analyzer=lambda toks: toks)
        X = vec.fit_transform([doc_tokens[d] for d in doc_ids])
        tfidf_runs = {}
        for q, q_toks in query_tokens.items():
            sims = cosine_similarity(vec.transform([q_toks]), X).ravel()
            idx = np.argsort(-sims, kind="stable")[:100]
            tfidf_runs[q] = [(doc_ids[i], float(sims[i])) for i in idx]

        bm25_api = BM25Okapi([doc_tokens[d] for d in doc_ids], k1=1.5, b=0.75)
        bm25_runs = {}
        for q, q_toks in query_tokens.items():
            scores = bm25_api.get_scores(q_toks)
            idx = np.argsort(-scores, kind="stable")[:100]
            bm25_runs[q] = [(doc_ids[i], float(scores[i])) for i in idx]

        runs = {"tfidf": tfidf_runs, "bm25": bm25_runs}
        with open(SPARSE_RUNS_PATH, "wb") as f:
            pickle.dump(runs, f)

runs["dense"] = dense_runs

if 'qrels_gain' not in globals():
    with open(os.path.join(DRIVE_DIR, "data.pkl"), "rb") as f:
        qrels_gain = pickle.load(f)["qrels_gain"]

p10 = lambda r: np.mean([sum(d in qrels_gain[q] for d,_ in r[q][:10])/10 for q in r])
r100 = lambda r: np.mean([sum(d in qrels_gain[q] for d,_ in r[q])/len(qrels_gain[q]) for q in r])
def mapk(r):
    out = []
    for q in r:
        hits = s = 0
        for i, (d, _) in enumerate(r[q], 1):
            if d in qrels_gain[q]: hits += 1; s += hits/i
        out.append(s/len(qrels_gain[q]))
    return np.mean(out)
def ndcg10(r):
    out = []
    for q in r:
        g = qrels_gain[q]
        dcg  = sum(g.get(d,0)/np.log2(i+1) for i,(d,_) in enumerate(r[q][:10],1))
        idcg = sum(x/np.log2(i+1) for i,x in enumerate(sorted(g.values(), reverse=True)[:10],1))
        out.append(dcg/idcg if idcg else 0)
    return np.mean(out)

runs["rrf_bm25_dense"] = rrf([runs["bm25"], runs["dense"]], k=60, top_k=100)
runs["rrf_all"] = rrf([runs["tfidf"], runs["bm25"], runs["dense"]], k=60, top_k=100)

for m, r in runs.items():
    print(f"{m:16} P@10={p10(r):.3f}  R@100={r100(r):.3f}  MAP={mapk(r):.3f}  nDCG@10={ndcg10(r):.3f}  queries={len(r)}")

In [ ]:
ALL_RUNS_OUT = "/content/drive/MyDrive/IR_project/all_runs.pkl"
os.makedirs(os.path.dirname(ALL_RUNS_OUT), exist_ok=True)
with open(ALL_RUNS_OUT, "wb") as f:
    pickle.dump(runs, f)
print("All runs saved to Google Drive:", ALL_RUNS_OUT)